# Silver Layer — tratamento dos dados

Neste notebook vamos ler as tabelas da camada Bronze e criar as tabelas da camada Silver. Nesta etapa faremos os tratamentos necessários nos dados, como a remoção de registros repetidos, a padronização dos textos, a conversão de datas e valores e o cruzamento entre as bases.

Como mais de uma pessoa do grupo pode executar o notebook, usamos `overwrite` para não acumular os mesmos registros mais de uma vez.

In [0]:
%run ./00_Config

In [0]:
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

In [0]:
dq_results = []


def dq_check(table_name, check_name, df, condition):
    total = df.count()
    failed = df.filter(~F.coalesce(condition, F.lit(False))).count()
    passed = failed == 0

    dq_results.append(
        Row(
            table_name=table_name,
            check_name=check_name,
            total_rows=total,
            failed_rows=failed,
            passed=passed,
            checked_at=datetime.now(),
        )
    )

    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")


def dq_check_unique(table_name, check_name, df, key_cols):
    total = df.count()
    duplicates = df.groupBy(*key_cols).count().filter(F.col("count") > 1).count()
    passed = duplicates == 0

    dq_results.append(
        Row(
            table_name=table_name,
            check_name=check_name,
            total_rows=total,
            failed_rows=duplicates,
            passed=passed,
            checked_at=datetime.now(),
        )
    )

    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {duplicates} chaves duplicadas")

## 1. Atendentes

Na base de atendentes vamos conferir se existe um registro por identificador, converter as datas e padronizar o status. O nível de atendimento aparece de formas diferentes, como `1`, `Nível 1` e `N2`, então esses valores também serão padronizados. A data de desligamento pode continuar nula quando o atendente ainda estiver ativo.

In [0]:
df_bronze_atendentes = spark.table(f"{bronze_schema}.tb_base_atendentes")

df_silver_atendentes = (
    df_bronze_atendentes
    .filter(F.col("id_atendente").isNotNull())
    .withColumn("id_atendente", F.trim(F.col("id_atendente")))
    .withColumn("nome_atendente", F.trim(F.col("nome_atendente")))
    .withColumn("nivel_atendimento", F.lower(F.trim(F.col("nivel_atendimento"))))
    .withColumn("nivel_atendimento",
        F.when(F.col("nivel_atendimento").isin("1", "nível 1"), 1)
        .when(F.col("nivel_atendimento").isin("2", "nível 2", "n2"), 2)
        .otherwise(None)
    )
    .withColumn("data_admissao", F.to_date(F.col("data_admissao"), "dd/MM/yyyy"))
    .withColumn("data_desligamento", F.to_date(F.col("data_desligamento"), "dd/MM/yyyy"))
    .withColumn("status_atendente", F.lower(F.trim(F.col("status_atendente"))))
)

dq_check_unique("silver.tb_atendentes", "id_atendente único", df_silver_atendentes, ["id_atendente"])
dq_check("silver.tb_atendentes", "nível de atendimento válido", df_silver_atendentes, F.col("nivel_atendimento").isNotNull() & F.col("nivel_atendimento").isin(1, 2))

df_silver_atendentes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_atendentes")

display(df_silver_atendentes.limit(10))

## 2. Motivos, canais e clientes

Essas três tabelas serão usadas como apoio nos próximos cruzamentos. Primeiro verificamos se as chaves são únicas; não removemos registros apenas pelo identificador, pois isso poderia escolher uma linha sem uma regra definida. A data de cadastro dos clientes aparece nos formatos `dd/MM/yyyy` e `yyyy-MM-dd`, então tentamos as duas conversões. As idades vazias ou iguais a zero serão mantidas e contabilizadas somente como diagnóstico, pois o case não define uma regra para corrigi-las.

In [0]:
df_bronze_motivos = spark.table(f"{bronze_schema}.tb_base_motivos")
df_bronze_canais = spark.table(f"{bronze_schema}.tb_canais")
df_bronze_clientes = spark.table(f"{bronze_schema}.tb_clientes")

mapa_motivos = F.create_map([
    F.lit("Transferência de agência"), F.lit("Cadastral"),
    F.lit("Cancelamento de conta"), F.lit("Cadastral"),
    F.lit("Contratação de cartão adicional"), F.lit("Cartão")
])

df_silver_motivos = (
    df_bronze_motivos
    .filter(F.col("id_motivo").isNotNull())
    .withColumn("id_motivo", F.trim(F.col("id_motivo")))
    .withColumn("nome_motivo", F.initcap(F.lower(F.trim(F.col("nome_motivo")))))
    .withColumn("categoria", F.coalesce(F.col("categoria"), mapa_motivos[F.col("nome_motivo")]))
    .withColumn("categoria", F.initcap(F.lower(F.trim(F.col("categoria")))))
    .withColumn("categoria", F.when(F.col("categoria") == "Cartao", "Cartão").otherwise(F.col("categoria")))
    .withColumn("criticidade", F.lower(F.trim(F.col("criticidade"))))
)

df_silver_canais = (
    df_bronze_canais
    .filter(F.col("id_canal").isNotNull())
    .withColumn("id_canal", F.trim(F.col("id_canal")))
    .withColumn("nome_canal", F.initcap(F.lower(F.trim(F.col("nome_canal")))))
    .withColumn("tipo_canal", F.initcap(F.lower(F.trim(F.col("tipo_canal")))))
    .withColumn("status_canal", F.lower(F.trim(F.col("status_canal"))))
)

df_silver_clientes = (
    df_bronze_clientes
    .filter(F.col("id_cliente").isNotNull())
    .withColumn("id_cliente", F.trim(F.col("id_cliente")))
    .withColumn("nome", F.trim(F.col("nome")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("regiao", F.initcap(F.lower(F.trim(F.col("regiao")))))
    .withColumn("idade", F.col("idade").cast("int"))
    .withColumn("data_cadastro",
        F.coalesce(
            F.try_to_date(F.col("data_cadastro"), "dd/MM/yyyy"),
            F.try_to_date(F.col("data_cadastro"), "yyyy-MM-dd"),
        )
    )
    .withColumn("dia_vencimento_fatura", F.col("dia_vencimento_fatura").cast("int"))
)

dq_check_unique("silver.tb_motivos", "id_motivo único", df_silver_motivos, ["id_motivo"])
dq_check_unique("silver.tb_canais", "id_canal único", df_silver_canais, ["id_canal"])
dq_check_unique("silver.tb_clientes", "id_cliente único", df_silver_clientes, ["id_cliente"])
dq_check("silver.tb_clientes", "data de cadastro convertida", df_silver_clientes, F.col("data_cadastro").isNotNull())
dq_check("silver.tb_clientes", "idade preenchida (diagnóstico)", df_silver_clientes, F.col("idade").isNotNull())
dq_check("silver.tb_clientes", "idade maior que zero (premissa de plausibilidade)", df_silver_clientes, F.col("idade").isNotNull() & (F.col("idade") > 0))

df_silver_motivos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_motivos")
df_silver_canais.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_canais")
df_silver_clientes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_clientes")

display(df_silver_motivos)
display(df_silver_canais)
display(df_silver_clientes.limit(10))

## 3. Horários dos chamados

Em `chamados.csv`, os campos de abertura e finalização estão vazios e o campo de início contém apenas textos como `igual a hora de abertura`. Por isso, usamos `Chamados_Hora.CSV` para obter os horários reais. Quando o início estiver vazio, o chamado será mantido com os tempos calculados nulos.

In [0]:
df_bronze_chamados_hora = spark.table(f"{bronze_schema}.tb_chamados_hora")

df_silver_chamados_hora = (
    df_bronze_chamados_hora
    .filter(F.col("ID_Chamado").isNotNull())
    .withColumnRenamed("ID_Chamado", "id_chamado")
    .withColumnRenamed("ID_Cliente", "id_cliente_horario")
    .withColumnRenamed("Hora_Abertura_Chamado", "hora_abertura_chamado")
    .withColumnRenamed("Hora_Inicio_Atendimento", "hora_inicio_atendimento")
    .withColumnRenamed("Hora_Finalizacao_Atendimento", "hora_finalizacao_atendimento")
    .withColumn("hora_abertura_chamado", F.to_timestamp(F.col("hora_abertura_chamado"), "dd/MM/yyyy 'às' HH:mm:ss"))
    .withColumn("hora_inicio_atendimento", F.to_timestamp(F.col("hora_inicio_atendimento"), "dd/MM/yyyy 'às' HH:mm:ss"))
    .withColumn("hora_finalizacao_atendimento", F.to_timestamp(F.col("hora_finalizacao_atendimento"), "dd/MM/yyyy 'às' HH:mm:ss"))
    .withColumn("tempo_espera_calculado",
        F.col("hora_inicio_atendimento").cast("long") - F.col("hora_abertura_chamado").cast("long"))
    .withColumn("tempo_atendimento_calculado",
        F.col("hora_finalizacao_atendimento").cast("long") - F.col("hora_inicio_atendimento").cast("long"))
)

dq_check_unique("silver.tb_chamados_hora", "id_chamado único", df_silver_chamados_hora, ["id_chamado"])
dq_check("silver.tb_chamados_hora", "horário de abertura convertido", df_silver_chamados_hora, F.col("hora_abertura_chamado").isNotNull())
dq_check("silver.tb_chamados_hora", "horário de início preenchido", df_silver_chamados_hora, F.col("hora_inicio_atendimento").isNotNull())
dq_check("silver.tb_chamados_hora", "horário de finalização convertido", df_silver_chamados_hora, F.col("hora_finalizacao_atendimento").isNotNull())
dq_check("silver.tb_chamados_hora", "tempo de espera não negativo", df_silver_chamados_hora, F.coalesce(F.col("tempo_espera_calculado") >= 0, F.lit(True)))
dq_check("silver.tb_chamados_hora", "tempo de atendimento não negativo", df_silver_chamados_hora, F.coalesce(F.col("tempo_atendimento_calculado") >= 0, F.lit(True)))

df_silver_chamados_hora.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_chamados_hora")

display(df_silver_chamados_hora.limit(10))

## 4. Chamados

Na base principal encontramos algumas linhas completamente repetidas, por isso usamos `dropDuplicates()`. Também padronizamos os valores de resolução, canal e motivo antes de fazer os joins com as tabelas de apoio. Os tempos informados e calculados são mantidos em colunas separadas para que suas diferenças apareçam nas verificações. Também conferimos se cliente e atendente existem nas respectivas bases e se o cliente é o mesmo nos dois arquivos de chamados.

In [0]:
df_bronze_chamados = spark.table(f"{bronze_schema}.tb_chamados")

df_silver_chamados = (
    df_bronze_chamados
    .dropDuplicates()
    .filter(F.col("id_chamado").isNotNull())
    .withColumn("motivo", F.lower(F.trim(F.col("motivo"))))
    .withColumn("canal", F.lower(F.trim(F.col("canal"))))
    .withColumn("canal", F.regexp_replace(F.col("canal"), r"[._]+", " "))
    .withColumn("canal", F.regexp_replace(F.col("canal"), r"\s+", " "))
    .withColumn("canal",
        F.when(F.col("canal").isin("u r a", "ura"), "ura")
        .when(F.col("canal").isin("chat bot", "chatbot"), "chatbot")
        .when(F.col("canal").isin("atend inicial", "atendimento inicial"), "atendimento inicial")
        .when(F.col("canal").isin("atend especializado", "atendimento especializado"), "atendimento especializado")
        .otherwise(F.col("canal"))
    )
    .withColumn("resolvido", F.lower(F.trim(F.col("resolvido"))))
    .withColumn("resolvido",
        F.when(F.col("resolvido").isin("sim", "s", "1"), "Sim")
        .when(F.col("resolvido").isin("não", "nao", "n", "0", "nao resolvido"), "Não")
        .otherwise("Não informado")
    )
    .withColumn("tempo_espera", F.when(F.upper(F.col("tempo_espera")) == "NULL", None).otherwise(F.col("tempo_espera")).cast("long"))
    .withColumn("tempo_atendimento", F.when(F.upper(F.col("tempo_atendimento")) == "NULL", None).otherwise(F.col("tempo_atendimento")).cast("long"))
    .withColumnRenamed("tempo_espera", "tempo_espera_informado")
    .withColumnRenamed("tempo_atendimento", "tempo_atendimento_informado")
    .drop("hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento")
)

df_motivos_join = df_silver_motivos.withColumn("motivo", F.lower(F.trim(F.col("nome_motivo"))))
df_canais_join = df_silver_canais.withColumn("canal", F.lower(F.trim(F.col("nome_canal"))))

df_clientes_referencia = df_silver_clientes.select("id_cliente").withColumn("cliente_cadastrado", F.lit(True))
df_atendentes_referencia = df_silver_atendentes.select("id_atendente").withColumn("atendente_cadastrado", F.lit(True))

df_silver_chamados = (
    df_silver_chamados
    .join(df_motivos_join.select("id_motivo", "motivo"), on="motivo", how="left")
    .join(df_canais_join.select("id_canal", "canal"), on="canal", how="left")
    .join(df_silver_chamados_hora, on="id_chamado", how="left")
    .join(df_clientes_referencia, on="id_cliente", how="left")
    .join(df_atendentes_referencia, on="id_atendente", how="left")
    .withColumn("cliente_cadastrado", F.coalesce(F.col("cliente_cadastrado"), F.lit(False)))
    .withColumn("atendente_cadastrado", F.coalesce(F.col("atendente_cadastrado"), F.lit(False)))
    .withColumn("cliente_horario_consistente", F.col("id_cliente") == F.col("id_cliente_horario"))
    .withColumn("tempo_espera_consistente", F.col("tempo_espera_informado") == F.col("tempo_espera_calculado"))
    .withColumn("tempo_atendimento_consistente", F.col("tempo_atendimento_informado") == F.col("tempo_atendimento_calculado"))
)

dq_check_unique("silver.tb_chamados", "id_chamado único", df_silver_chamados, ["id_chamado"])
dq_check("silver.tb_chamados", "motivo identificado", df_silver_chamados, F.col("id_motivo").isNotNull())
dq_check("silver.tb_chamados", "canal identificado", df_silver_chamados, F.col("id_canal").isNotNull())
dq_check("silver.tb_chamados", "cliente encontrado na base de clientes", df_silver_chamados, F.col("cliente_cadastrado"))
dq_check("silver.tb_chamados", "atendente encontrado na base de atendentes", df_silver_chamados, F.col("atendente_cadastrado"))
dq_check("silver.tb_chamados", "cliente igual nas duas bases de chamados", df_silver_chamados, F.coalesce(F.col("cliente_horario_consistente"), F.lit(False)))
dq_check("silver.tb_chamados", "tempo de espera consistente quando ambos estão disponíveis", df_silver_chamados, F.coalesce(F.col("tempo_espera_consistente"), F.lit(True)))
dq_check("silver.tb_chamados", "tempo de atendimento consistente quando ambos estão disponíveis", df_silver_chamados, F.coalesce(F.col("tempo_atendimento_consistente"), F.lit(True)))

df_silver_chamados.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_chamados")

display(df_silver_chamados.limit(10))

## 5. Custos e pesquisa de satisfação

Nos custos removemos apenas as linhas completamente repetidas e retiramos os textos `R$` e `reais` antes da conversão para decimal. Valores negativos serão mantidos, pois o case não informa se eles representam erro, estorno ou ajuste. Nas pesquisas mantemos as respostas recebidas e criamos uma coluna para indicar se a nota está entre 1 e 5, conforme a descrição da base no enunciado.

In [0]:
df_bronze_custos = spark.table(f"{bronze_schema}.tb_custos")
df_bronze_pesquisas = spark.table(f"{bronze_schema}.tb_pesquisa_satisfacao")

df_silver_custos = (
    df_bronze_custos
    .dropDuplicates()
    .filter(
        F.col("id_custo").isNotNull()
        & F.col("id_chamado").isNotNull()
    )
    .withColumn(
        "custo",
        F.lower(F.trim(F.col("custo")))
    )
    .withColumn(
        "custo",
        F.replace(
            F.col("custo"),
            F.lit("r$"),
            F.lit("")
        )
    )
    .withColumn(
        "custo",
        F.replace(
            F.col("custo"),
            F.lit("reais"),
            F.lit("")
        )
    )
    .withColumn(
        "custo",
        F.replace(
            F.col("custo"),
            F.lit(","),
            F.lit(".")
        )
    )
    .withColumn(
        "custo",
        F.trim(F.col("custo"))
    )
    .withColumn(
        "custo",
        F.col("custo").cast("decimal(10,2)")
    )
)

df_silver_pesquisas = (
    df_bronze_pesquisas
    .filter(F.col("id_pesquisa").isNotNull() & F.col("id_chamado").isNotNull())
    .withColumn("nota_atendimento",
        F.when(F.trim(F.col("nota_atendimento")) == "", None)
        .when(F.upper(F.trim(F.col("nota_atendimento"))) == "NULL", None)
        .otherwise(F.col("nota_atendimento"))
        .cast("int")
    )
    .withColumn("nota_valida", F.col("nota_atendimento").between(1, 5))
)

dq_check_unique("silver.tb_custos", "id_custo único", df_silver_custos, ["id_custo"])
dq_check("silver.tb_custos", "custo convertido para decimal", df_silver_custos, F.col("custo").isNotNull())
dq_check_unique("silver.tb_pesquisa_satisfacao", "id_pesquisa único", df_silver_pesquisas, ["id_pesquisa"])
dq_check("silver.tb_pesquisa_satisfacao", "nota entre 1 e 5", df_silver_pesquisas, F.coalesce(F.col("nota_valida"), F.lit(False)))

df_silver_custos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_custos")
df_silver_pesquisas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_pesquisa_satisfacao")

display(df_silver_custos.limit(10))
display(df_silver_pesquisas.limit(10))

## 6. Resultado das verificações

As verificações realizadas durante o notebook são reunidas em uma tabela para facilitar a revisão do grupo.

In [0]:
df_dq_log = spark.createDataFrame(dq_results)

df_dq_log.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.dq_log")

display(df_dq_log)